## config

In [0]:
%run ../config

## Objective

In [0]:
sales_df = read_data(path=bronze_layer['sales_data'],type='csv')\
            .select('date','store_nbr','family','sales','onpromotion')
sales_df.limit(5).display()


In [0]:
print(f"number of records : {sales_df.count()}")
print(f"number of unique stores {sales_df.select('store_nbr').distinct().count()}")
print(f"number of unique families {sales_df.select('family').distinct().count()}")
print(f"data is from {sales_df.agg(f.min('date')).collect()[0][0]} to {sales_df.agg(f.max('date')).collect()[0][0]}")

## Data Sanity checks

### S1.missing days

In [0]:

min_date = sales_df.agg(F.min('date')).collect()[0][0]
max_date = sales_df.agg(F.max('date')).collect()[0][0]
days = (max_date-min_date).days + 1
print(min_date, max_date,days)

sales_df.groupBy('store_nbr','family').agg(f.countDistinct('date').alias('date_c'),F.min('date'),F.max('date')).display()

In [0]:
full_dates_set = set([str(datetime.date(d)) for d in pd.date_range(start=min_date, end=max_date)])
prd_str_dates = set([str(d['date']) for d in sales_df.filter(f.col('store_nbr')==20).filter(f.col('family')=="CLEANING").select('date').distinct().collect()])
print(full_dates_set - prd_str_dates)

- On Christmas the stores are closed,this might the reason for no sales in the sales data.

### S2.negative sales

In [0]:
sales_df.filter(f.col('sales')<0).display()

- no negative sales

## week fill

In [0]:
min_date = '2014-01-01'
max_date = '2017-07-31'

sales_df1 = sales_df.filter(f.col('date')>=min_date).filter(F.col('date')<=max_date)

In [0]:
# sales_df1.groupby('family','store_nbr').agg(f.countDistinct('date')).display() count:1305

In [0]:
dates_df = spark.createDataFrame(data=[str(datetime.date(d)) for d in pd.date_range(start=min_date, end=max_date)],schema=['date'])
products_df = sales_df.select('family').distinct()
customers_df = sales_df.select('store_nbr').distinct()

full_sales_data = dates_df.crossJoin(products_df).crossJoin(customers_df)
full_sales_data.display()

In [0]:
# dates_df.count(),full_sales_data.groupby('family','store_nbr').agg(f.countDistinct('date')).display() #1308

In [0]:
dates_filled_data = full_sales_data.join(sales_df1,on=['date','family','store_nbr'],how='left').fillna(0,['sales','onpromotion'])
dates_filled_data.filter(f.col('sales')<0).display()

## initial zero removal

In [0]:
min_non_zero_sales_date = dates_filled_data.filter(f.col('sales')>0).groupby('family','store_nbr').agg(f.min('date').alias('min_non_zero_sales_date'))
initial_zero_removal = dates_filled_data.join(min_non_zero_sales_date,on=['family','store_nbr'],how='left').filter(f.col('date')>F.col('min_non_zero_sales_date')).drop('min_non_zero_sales_date')
initial_zero_removal.display()


In [0]:
initial_zero_removal.filter(F.col('family')=="SCHOOL AND OFFICE SUPPLIES").filter(F.col('store_nbr')==52).display()

Databricks visualization. Run in Databricks to view.

## oil data process and merger

In [0]:
oil_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load(bronze_layer['oil_data'])
# oil_df.limit(5).display()

sales_oil_joined = initial_zero_removal.join(oil_df,on=['date'],how='left')
window_spec = Window.partitionBy('family','store_nbr').orderBy(F.col('date').asc()).rowsBetween(Window.unboundedPreceding, 0)

sales_oil_joined1 = sales_oil_joined.withColumn('oil_price',F.last('dcoilwtico',ignorenulls=True).over(window_spec)).drop('dcoilwtico')
assert sales_oil_joined1.count() == initial_zero_removal.count(),"Duplicates!!"

## store data process and merger

In [0]:
stores_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load(bronze_layer['stores_data'])
# stores_df.display()

stores_joined = sales_oil_joined1.join(stores_df,on=['store_nbr'],how='left')
print(sales_oil_joined.count(),stores_joined.count())
stores_joined.display()

## holiday data process and merger

In [0]:
holiday_data = spark.read.format('csv').option('header',True).option('inferSchema',True).load(bronze_layer['holiday_data'])
holiday_data.display()

In [0]:
# holiday_data.agg(F.min('date'),f.max('date'),F.countDistinct('locale_name'),F.countDistinct('type'),F.countDistinct('locale'),F.countDistinct('description')).display()
# holiday_data.select('type').distinct().display()
# holiday_data.select('locale').distinct().display()
# holiday_data.select('locale_name').distinct().display()


In [0]:
holidays = [h['locale_name'] for h in holiday_data.filter(f.col('type')=="Holiday").select('locale_name').distinct().collect()]
# events = [h['locale_name'] for h in holiday_data.filter(f.col('type')=="Event").select('locale_name').distinct().collect()]
# holidays

In [0]:
for h in holidays:
    holiday_data = holiday_data.withColumn(h,F.when(f.col('locale_name')==h,F.lit(1)).otherwise(F.lit(0)))
# holiday_data = holiday_data.withColumn('Ecuador',F.when(F.col('locale_name')=='Ecuador',F.lit(1)).otherwise(F.lit(0)))

holiday_data_encoded = holiday_data.groupby('date').agg(*[F.max(h).alias(h) for h in holidays])
holiday_data_encoded.display()

In [0]:
holiday_cols = [str(h) for h in  holiday_data_encoded.columns if h not in ['date']]
new_holiday_cols = [f"f_holiday_{h.strip().replace(" ","_").lower()}" for h in holiday_cols]

for old_col, new_col in zip(holiday_cols,new_holiday_cols):
    holiday_data_encoded = holiday_data_encoded.withColumnRenamed(old_col,new_col)


In [0]:
holiday_data_encoded.display()

In [0]:
holiday_joined = stores_joined.join(holiday_data_encoded,on=['date'],how='left').fillna(0)
print(holiday_joined.count(),stores_joined.count())

## transcation data process and merger

In [0]:
transcation_df = spark.read.format('csv').option('header',True).option('inferSchema',True).load(bronze_layer['transcations_data'])
transcation_df.display()

## modelling data

In [0]:
final_modelling_data = holiday_joined.join(transcation_df,on=['store_nbr','date'],how='left')
final_modelling_data.display()

## save modelling data

In [0]:
silver_layer['modelling_data']

In [0]:
final_modelling_data.write.format('parquet').mode('overwrite').save(silver_layer['modelling_data'])

In [0]:
spark.read.parquet(silver_layer['modelling_data']).count()